# HoloForge Part 2 -- Colab GPU runner (one-cell, auto-resuming)

**Two cells total.** Run cell 1 once to confirm the GPU, then run cell 2 -- it does everything: mounts Drive, clones/pulls the repo, installs dependencies, runs the compute-budget probe (once), and then runs E1-E5 to completion, auto-resuming between internal time-chunks with no extra clicks.

**If Colab disconnects you** (free-tier idle/session limits -- this notebook cannot prevent that, it's a platform limit): reconnect, then just **re-run cell 2 again**. It is fully idempotent and self-contained -- it does not depend on any variable from a previous run surviving the disconnect. Already-finished jobs are stored on Google Drive (not Colab's ephemeral local disk) and are detected and skipped automatically, so you never lose more than the one job that was actually in flight when the runtime died.

**Gate 1**: cell 2 probes compute cost once (results cached in a marker file on Drive) and only proceeds to the full run if it's under the 40 T4-hour threshold. If it's over, the cell stops and prints your reduction options -- it will not silently pick one for you.

In [ ]:
!nvidia-smi
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU -- fix Runtime > Change runtime type before continuing."

In [ ]:
import os, sys, importlib

# ---- 1. Mount Drive (idempotent). Every atomic result write lands here,
# not on Colab's ephemeral local disk, so a disconnect never loses
# finished work -- only whatever job was still running when it died. ----
from google.colab import drive
drive.mount('/content/drive')
RESULTS_DIR = '/content/drive/MyDrive/holoforge_results'
os.makedirs(RESULTS_DIR, exist_ok=True)

# ---- 2. Clone or pull the repo (idempotent) ----
REPO_DIR = '/content/HoloForge'
BRANCH = 'media-part2-followups'
if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    print("repo already present, pulling latest...")
    !cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}
else:
    !git clone -b {BRANCH} https://github.com/ultramagnus23/HoloForge.git {REPO_DIR}

WORKDIR = os.path.join(REPO_DIR, 'holography_media')
os.chdir(WORKDIR)
!git rev-parse HEAD

# ---- 3. Install dependencies (idempotent) ----
!pip install -q scipy torcwa

# ---- 4. Import the runner FRESH every time this cell runs -- sys.path and
# module state don't survive a runtime disconnect either, so this cannot
# assume anything from a previous execution. ----
for p in [WORKDIR, os.path.join(WORKDIR, 'experiments')]:
    if p not in sys.path:
        sys.path.insert(0, p)
if 'run_manifest' in sys.modules:
    importlib.reload(sys.modules['run_manifest'])
import run_manifest as rm
rm.set_results_root(RESULTS_DIR)
import torch
torch.set_default_dtype(torch.float64)

# ---- 5. Probe + Gate 1 -- ONLY on the first run. A marker file on Drive
# (survives disconnects) records that it already passed, so a resume after
# a disconnect skips straight to the real work instead of re-probing for
# several minutes every single reconnect. ----
GATE1_MARKER = os.path.join(RESULTS_DIR, '_gate1_passed.marker')
if os.path.exists(GATE1_MARKER):
    print(f"Gate 1 already passed in a previous run ({open(GATE1_MARKER).read().strip()}) "
         f"-- skipping re-probe, resuming the real work directly.")
    proceed = True
else:
    grand_total = rm.probe('all')
    if grand_total > rm.GATE1_HOURS:
        proceed = False
        print("\n" + "="*70)
        print(f"GATE 1 FAILED ({grand_total:.1f}h > {rm.GATE1_HOURS:.0f}h) -- stopping here.")
        print("This does NOT auto-reduce scope -- review the table printed above and")
        print("decide (fewer seeds / coarser grid / drop E6), then have that reflected")
        print("in experiments/manifest.py before re-running this cell.")
        print("="*70)
    else:
        proceed = True
        with open(GATE1_MARKER, 'w') as f:
            f.write(f"{grand_total:.1f}h, passed")
        print(f"\nGATE 1 passed ({grand_total:.1f}h <= {rm.GATE1_HOURS:.0f}h) -- "
             f"running the full manifests now.\n")

# ---- 6. Auto-resuming run. Chunked so a mid-chunk Colab kill never loses
# more than one job; re-running this whole cell after a reconnect resumes
# automatically (steps 1-5 above are all idempotent / marker-gated). ----
if proceed:
    CHUNK_MINUTES = 60
    for manifest_name in ["E1", "E2", "E3", "E4", "E5"]:
        print(f"\n{'='*20} {manifest_name} {'='*20}")
        while True:
            status = rm.run_manifest(manifest_name, max_minutes=CHUNK_MINUTES)
            if status["complete"]:
                print(f"[{manifest_name}] COMPLETE ({status['n_total']} jobs).")
                break
            # chunk's time ran out with jobs still remaining -- loop calls
            # run_manifest again immediately, which resumes (skips already-
            # done jobs) with no manual re-click, as long as this Colab
            # runtime is still alive. If it dies here, re-run this cell.
    print("\nAll manifests attempted -- see per-manifest status above.")
    print("E7 needs no GPU; run separately any time: !python experiments/rcwa_crosscheck.py e7")

## When it's done: bring results back into the repo

Results live on Drive at `RESULTS_DIR` the whole time (that's what makes the auto-resume safe). Once you're satisfied a manifest is complete, run this to copy it into the cloned repo and print the exact git command -- it deliberately does not push automatically.

In [ ]:
import shutil
for manifest_name in ["E1", "E2", "E3", "E4", "E5"]:
    src = os.path.join(RESULTS_DIR, manifest_name)
    dst = os.path.join(REPO_DIR, 'holography_media', 'results', manifest_name)
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"copied {src} -> {dst}")
print("\nNow (from Colab, if you've set up git credentials, or after downloading):")
print(f"  !cd {REPO_DIR} && git add holography_media/results/ && git commit -m '...' && git push")